In [118]:
import requests
import json
import os
import time
from bs4 import BeautifulSoup
import newspaper
from newspaper import Config


In [119]:
def load_saved_links(file_path):
    """Load saved links from a JSON array file."""
    saved_links = set()
    if os.path.exists(file_path):  # Check if the file exists
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)  # Load the entire JSON array
                for entry in data:
                    if 'link' in entry:
                        saved_links.add(entry['link'])  # Add the link to the set
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON file: {e}")
    return saved_links

def fetch_search_results(url, headers, retries=3):
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except requests.exceptions.RequestException as e:
            print(f"Error fetching search results: {e}")
            time.sleep(2)  # Wait before retrying
    return None

def extract_results(soup, limit=10):
    """
    Extract the first `limit` search result links from the BeautifulSoup-parsed page.
    Note: You might need to adjust the selector if Bing's page structure changes.
    """
    return soup.find_all('a', class_='title', limit=limit) if soup else [] # how many articles to extract

def extract_article_with_newspaper(link, user_agent):
    """
    Use the newspaper library with a custom user agent to download, parse, and extract article content.
    Returns the full text and a summary (first 50 words).
    """
    try:
        # Create a configuration object and set your custom browser user agent.
        config = Config()
        config.browser_user_agent = user_agent
        article = newspaper.Article(link, config=config)
        article.download()
        article.parse()
        text = article.text
        summary = ' '.join(text.split()[:50])  # first 50 words as a summary
        
        # Attempt to get the publish date.
        publish_date = article.publish_date
        if publish_date:
            # Format the date as a string; adjust the format if needed.
            publish_date_str = publish_date.strftime("%Y-%m-%d %H:%M:%S")
        else:
            publish_date_str = "Unknown"
            
        return text, summary, publish_date_str
    except Exception as e:
        print(f"Error extracting article from {link}: {e}")
        time.sleep(2)  # Sleep to avoid overwhelming the server
        return None, None, None

def fallback_extraction(link, headers):
    """
    Fallback extraction using BeautifulSoup.
    This method attempts to extract the page content by targeting common content containers.
    """
    try:
        response = requests.get(link, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        # Try to locate common container elements for article text.
        article_div = soup.find('div', class_='article-body') or soup.find('div', class_='content')
        if article_div:
            text = article_div.get_text(separator=' ', strip=True)
        else:
            # Fallback to extracting all text from the page.
            text = soup.get_text(separator=' ', strip=True)
        summary = ' '.join(text.split()[:50])
        publish_date_str = "Unknown"
        return text, summary, publish_date_str
    except Exception as e:
        print(f"Fallback extraction failed for {link}: {e}")
        return None, None, None 

def extract_article(link, user_agent, headers):
    """
    Attempts to extract article content using Newspaper.
    If extraction fails or the content is too short, fallback to BeautifulSoup extraction.
    """
    text, summary, publish_date = extract_article_with_newspaper(link, user_agent)
    # Use a threshold of 100 words; adjust this value based on your requirements.
    if not text or len(text.split()) < 100:
        print("Falling back to BeautifulSoup extraction due to insufficient content length.")
        text, summary, publish_date = fallback_extraction(link, headers)
    return text, summary, publish_date

def save_to_json(data, file_path):
    """Save the provided data into a JSON file as part of a JSON array."""
    try:
        # Check if the file exists and load existing data
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    existing_data = json.load(f)  # Load the existing JSON array
                except json.JSONDecodeError:
                    existing_data = []  # If the file is empty or invalid, start with an empty list
        else:
            existing_data = []

        # Append the new data to the existing array
        existing_data.append(data)

        # Write the updated array back to the file
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(existing_data, f, indent=4, ensure_ascii=False)

    except IOError as e:
        print(f"Error saving to file: {e}")

def main():
    # Define a list of search terms.
    search_terms = ["Cybersecurity"]

    # Set your HTTP request headers; these help mimic a real browser.
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    }
    user_agent = headers['User-Agent']
    
    # Define the output JSON file path.
    file_path = '/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/BingSearchData.json'

    # Load saved links from the JSON file.
    saved_links = load_saved_links(file_path)
    print(f"Loaded {len(saved_links)} saved links from previous runs.")

    # Desired number of valid articles per search term.
    desired_valid_articles = 10

    for term in search_terms:
        print(f"\nSearching for: {term}")
        search_url = f"https://www.bing.com/news/search?q={term}"
        soup = fetch_search_results(search_url, headers)
        # Increase candidate limit to allow for some skipped links.
        candidate_links = extract_results(soup, limit=20)
        
        valid_articles_count = 0  # Reset count for each search term.
        seen_links = set()  # Initialize the set to track processed links.

        for result in candidate_links:
            link = result.get('href')

            # Skip duplicate links (both in current session and previously saved links)
            if link in seen_links or link in saved_links:
                continue
            seen_links.add(link)

            title = result.get_text(strip=True)
            print(f"Title: {title}")
            print(f"Link: {link}")
            timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

            # Attempt to extract the article with the combined method.
            content, summary, publish_date = extract_article(link, user_agent, headers)
            
            # Only count articles with sufficient content (at least 100 words).
            if content and len(content.split()) >= 100:
                print(f"Summary: {summary}")
                data = {
                    'search_term': term,
                    'timestamp': timestamp,
                    'title': title,
                    'link': link,
                    'content': content,
                    'summary': summary,
                    'publish_date': publish_date
                }
            
                try:
                    # Save the data in JSON array format
                    save_to_json(data, file_path)
            
                    # Add the link to the saved links set
                    saved_links.add(link)
                    valid_articles_count += 1
                except (TypeError, ValueError) as e:
                    print(f"Error saving data for link {link}: {e}")
                print(f"Valid articles so far: {valid_articles_count}/{desired_valid_articles}")
                # Stop processing candidate links once we've reached the desired count.
                if valid_articles_count >= desired_valid_articles:
                    break
            else:
                print("Excluding this link due to insufficient extracted content.")
                
        if valid_articles_count < desired_valid_articles:
            print(f"Only {valid_articles_count} valid articles were found for search term '{term}'.")

if __name__ == "__main__":
    main()


Loaded 13 saved links from previous runs.

Searching for: Cybersecurity
Title: Leadership In Cybersecurity Disruption: Turning Emerging Threats Into Competitive Opportunities
Link: https://www.forbes.com/councils/forbestechcouncil/2025/04/09/leadership-in-cybersecurity-disruption-turning-emerging-threats-into-competitive-opportunities/
Summary: Francis Dinha is CEO and cofounder of OpenVPN Inc., a leading enterprise network security company. getty In today's digital landscape, cybersecurity threats are no longer just an IT issue; they have evolved into a core business challenge that demands executive attention at the highest levels. The rapid rise of ransomware
Valid articles so far: 1/10
Title: 8 simple ways to teach your friends and family about cybersecurity - before it's too late
Link: https://www.zdnet.com/article/8-simple-ways-to-teach-your-friends-and-family-about-cybersecurity-before-its-too-late/
Summary: LagartoFilm/Getty Images It's a jungle out there. And by "out there," I 